### LAB | Whisper STT Implementation

In [2]:
#pip install openai pydub

**Note:** sudo apt-get install ffmpeg

In [1]:
import sounddevice as sd
import numpy as np
import scipy.io.wavfile as wavfile
import librosa
import librosa.display
import matplotlib.pyplot as plt
from IPython.display import Audio, display

from openai import OpenAI
import io
import os
from IPython.display import Audio, display
from dotenv import load_dotenv

### Basic Transcript (Without Chunking)

In [2]:
audio_path="Boatmaking.mp3"

# Load only the first 30 seconds, resampled to 16 kHz mono (what Whisper expects)
audio_sample, sr = librosa.load(audio_path, sr=16000, mono=True, offset=0.0, duration=30.0)

audio_sample=audio_sample.flatten()

/home/fatmamerayy/miniconda3/envs/bootcamp-env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))


audio_int16 = np.int16(audio_sample * 32767)


# Create in-memory WAV file
sample_rate=16000
buffer = io.BytesIO()

wavfile.write(buffer, sample_rate, audio_int16)
buffer.seek(0)
buffer.name = "recording.wav"  # Whisper needs a filename

# Transcribe with Whisper
print("🤖 Transcribing with Whisper...")
transcript = client.audio.transcriptions.create(
    model="whisper-1",
    file=buffer
)

print("Transcription result:",transcript.text)


🤖 Transcribing with Whisper...
Transcription result: Well, how do you go about making a small rowboat? We just make the small scale model and draft it from that. Make a keel out. You make a scale model first? Most everybody does, make a scale model. Or else they draft them out, draw them out on paper. Either one you want to, it doesn't matter. How big are these scale models? A general rule on small type.


### Transcript with Prompt

In [4]:
# Record for 30 seconds
duration = 15
sample_rate = 16000

print(f"\n🎤 Recording for {duration} seconds... Start reading now!")
audio = sd.rec(int(duration * sample_rate), samplerate=sample_rate, channels=1, dtype='float32')
sd.wait()
audio = audio.flatten()
print("✅ Recording complete!")

display(Audio(audio, rate=sample_rate))

# Prepare audio
audio_int16 = np.int16(audio * 32767)
buffer = io.BytesIO()
wavfile.write(buffer, sample_rate, audio_int16)
buffer.seek(0)
buffer.name = "recording.wav"

# Transcribe without context prompt
transcript = client.audio.transcriptions.create(
    model="whisper-1",
    file=buffer,
)

print(transcript.text)


🎤 Recording for 15 seconds... Start reading now!
✅ Recording complete!


/home/fatmamerayy/miniconda3/envs/bootcamp-env/lib/python3.11/site-packages/IPython/lib/display.py:186: RuntimeWarning: invalid value encountered in divide
  scaled = data / normalization_factor * 32767
/home/fatmamerayy/miniconda3/envs/bootcamp-env/lib/python3.11/site-packages/IPython/lib/display.py:187: RuntimeWarning: invalid value encountered in cast
  return scaled.astype("<h").tobytes(), nchan


you


In [5]:
# Transcribe with context prompt
transcript = client.audio.transcriptions.create(
    model="whisper-1",
    file=buffer,
    prompt = "I'm speaking about how to make rowboats. Please transcribe my speech accurately and clearly, including any technical terms related to boatmaking."
)

print("Transcription result:",transcript.text)

Transcription result: 


### Implementing Audio Chunking

In [6]:
audio, sr = librosa.load(audio_path, sr=16000, mono=True)

audio=audio.flatten()

In [7]:
chunk_duration = 5 
chunk_size = chunk_duration * sample_rate
chunks = [audio[i:i+chunk_size] for i in range(0, len(audio), chunk_size)]

print(f"Split into {len(chunks)} chunks")

Split into 20 chunks


In [8]:
all_transcripts = []


for i, chunk in enumerate(chunks):
    print(f"\n🤖 Transcribing chunk {i+1}/{len(chunks)}...")
    
    # Prepare chunk buffer
    chunk_int16 = np.int16(chunk * 32767)
    buffer = io.BytesIO()
    wavfile.write(buffer, sample_rate, chunk_int16)
    buffer.seek(0)
    buffer.name = f"chunk_{i}.wav"
    
    # Transcribe
    transcript = client.audio.transcriptions.create(
        model="whisper-1",
        file=buffer
    )
    
    all_transcripts.append(transcript.text)
    print(f"Chunk {i+1}: {transcript.text}")


🤖 Transcribing chunk 1/20...
Chunk 1: Well, how do you go about making a, uh, small robot? What does that make?

🤖 Transcribing chunk 2/20...
Chunk 2: the small-scale model and drafted from that.

🤖 Transcribing chunk 3/20...
Chunk 3: Make a keel out, you make a scale model first.

🤖 Transcribing chunk 4/20...
Chunk 4: Most everybody does, I guess, be a one, or I'll say draft them out.

🤖 Transcribing chunk 5/20...
Chunk 5: Draw the model on paper, either one you want to do, it doesn't matter.

🤖 Transcribing chunk 6/20...
Chunk 6: How big are these scale models? A general rule on small type.

🤖 Transcribing chunk 7/20...
Chunk 7: Oh, just a three-quarter inch to a foot.

🤖 Transcribing chunk 8/20...
Chunk 8: The large ones are up to, uh...

🤖 Transcribing chunk 9/20...
Chunk 9: quarter-inch to a fourth and what's the purpose of the

🤖 Transcribing chunk 10/20...
Chunk 10: scales, scale models. Well, the length, and the width, and the width.

🤖 Transcribing chunk 11/20...
Chunk 11: Oh

In [9]:
full_text = " ".join(all_transcripts)
print(full_text)

Well, how do you go about making a, uh, small robot? What does that make? the small-scale model and drafted from that. Make a keel out, you make a scale model first. Most everybody does, I guess, be a one, or I'll say draft them out. Draw the model on paper, either one you want to do, it doesn't matter. How big are these scale models? A general rule on small type. Oh, just a three-quarter inch to a foot. The large ones are up to, uh... quarter-inch to a fourth and what's the purpose of the scales, scale models. Well, the length, and the width, and the width. Oh, I see. The tissue is smaller than everything else. Then they just scale them up and down. And how do you go about deciding to build the boat itself? Well, you make a key. first from the model, from the drafting, drawing, what I did. and you've got the stem and the stern. And for the small stuff, the small bolts, where you bend the frame. byddwch chi'n gwneud môl, mae'n ymwneud â'r môl, felly mae'n rhan o'i gilydd. of the at the

### Transcribing Chunks with Timestamps

In [11]:
audio, sr = librosa.load(audio_path, sr=16000, mono=True)

In [13]:
def fmt(t):
    """Seconds -> HH:MM:SS.mmm"""
    h, rem = divmod(t, 3600)
    m, s = divmod(rem, 60)
    return f"{int(h):02d}:{int(m):02d}:{s:06.3f}"

all_segments = []   # every segment with global timestamps
chunk_start = 0.0   # start time of the current chunk, in seconds

for i, chunk in enumerate(chunks):
    # Prepare chunk buffer
    chunk_int16 = np.int16(chunk * 32767)
    buffer = io.BytesIO()
    wavfile.write(buffer, sample_rate, chunk_int16)
    buffer.seek(0)
    buffer.name = f"chunk_{i}.wav"

    # Transcribe with timestamps
    transcript = client.audio.transcriptions.create(
        model="whisper-1",
        file=buffer,
        response_format="verbose_json",
        timestamp_granularities=["segment"],
    )

    # Shift each segment by the chunk's start time
    for seg in transcript.segments:
        all_segments.append({
            "start": chunk_start + seg.start,
            "end": chunk_start + seg.end,
            "text": seg.text.strip(),
        })

    # Advance the offset by this chunk's real duration
    chunk_start += len(chunk) / sample_rate

# Combined transcript with timestamps
full_text = "\n".join(
    f"[{fmt(s['start'])} --> {fmt(s['end'])}] {s['text']}"
    for s in all_segments
)
print(full_text)

[00:00:00.000 --> 00:00:03.000] Well, how do you go about making a, uh, small robot?
[00:00:04.000 --> 00:00:05.000] What does that make?
[00:00:05.000 --> 00:00:09.520] the small-scale model and drafted from that.
[00:00:10.000 --> 00:00:14.600] Make a keel out, you make a scale model first.
[00:00:15.000 --> 00:00:20.000] Most everybody does, I guess, be a one, or I'll say draft them out.
[00:00:20.000 --> 00:00:25.000] Draw the model on paper, either one you want to do, it doesn't matter.
[00:00:25.000 --> 00:00:28.080] How big are these scale models?
[00:00:28.080 --> 00:00:29.840] A general rule on small type.
[00:00:30.000 --> 00:00:34.240] Oh, just a three-quarter inch to a foot.
[00:00:35.000 --> 00:00:39.880] The large ones are up to, uh...
[00:00:40.000 --> 00:00:46.720] quarter-inch to a fourth and what's the purpose of the
[00:00:45.000 --> 00:00:47.500] scales, scale models.
[00:00:47.500 --> 00:00:50.000] Well, the length, and the width, and the width.
[00:00:50.000 --> 0

### Exporting with Timestamps

In [15]:
import json

def fmt_srt(t):
    """Seconds -> HH:MM:SS,mmm (SRT uses a comma)"""
    ms = int(round(t * 1000))
    h, ms = divmod(ms, 3_600_000)
    m, ms = divmod(ms, 60_000)
    s, ms = divmod(ms, 1000)
    return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"

def fmt_short(t):
    """Seconds -> MM:SS (or HH:MM:SS if over an hour)"""
    t = int(t)
    h, rem = divmod(t, 3600)
    m, s = divmod(rem, 60)
    return f"{h:02d}:{m:02d}:{s:02d}" if h else f"{m:02d}:{s:02d}"

In [17]:
# 1. Human-readable text file
with open("transcript.txt", "w", encoding="utf-8") as f:
    for seg in all_segments:
        f.write(f"[{fmt_short(seg['start'])} - {fmt_short(seg['end'])}] {seg['text']}\n")

# 2. SRT subtitle file
with open("transcript.srt", "w", encoding="utf-8") as f:
    for i, seg in enumerate(all_segments, start=1):
        f.write(f"{i}\n")
        f.write(f"{fmt_srt(seg['start'])} --> {fmt_srt(seg['end'])}\n")
        f.write(f"{seg['text']}\n\n")
        
# 3. JSON file
output = {
    "duration": all_segments[-1]["end"] if all_segments else 0,
    "full_text": " ".join(s["text"] for s in all_segments),
    "segments": [
        {
            "id": i,
            "start": round(s["start"], 3),
            "end": round(s["end"], 3),
            "text": s["text"],
        }
        for i, s in enumerate(all_segments)
    ],
}
with open("transcript.json", "w", encoding="utf-8") as f:
    json.dump(output, f, ensure_ascii=False, indent=2)

print("Saved: transcript.txt, transcript.srt, transcript.json")        

Saved: transcript.txt, transcript.srt, transcript.json
